# Python Basics Part 2 — Theory & Reference Companion

This notebook is the **theory companion** to `python_course.ipynb`. It follows
the exact same order, section by section, but instead of "run this cell and
watch," it explains *why* things work the way they do and gives you tables to
look things up later. Keep the two notebooks side by side: run the examples
over there, look up the rules over here.

## 1 — Decisions with `if`

`if` runs an indented block only when its condition is `True`. Python uses
**indentation** (by convention, four spaces) to mark what belongs to the
block — there are no curly braces.

```
if condition_1:
    ...            # runs only if condition_1 is True
elif condition_2:
    ...            # runs only if condition_1 was False and condition_2 is True
else:
    ...            # runs if nothing above matched
```

Branches are checked **top to bottom**, and the **first** one that matches
wins — later branches are skipped even if they would also be true. `elif`
and `else` are both optional; you can have zero or many `elif`s.

## 2 — `for` loops

A `for` loop walks through a collection one item at a time. The loop
variable takes each value in turn, and the indented body runs once per item.

```
for item in collection:
    ...   # runs once per item, with `item` bound to the current one
```

### Building blocks that combine with `for`

| Tool | Purpose | Example | Produces |
|---|---|---|---|
| Running counter | Track how many / which number we're on | `count = count + 1` inside the loop | A manually incremented tally |
| `range(a, b)` | Generate whole numbers | `range(0, 5)` | `0, 1, 2, 3, 4` (half-open, like slicing) |
| `enumerate(collection)` | Get index **and** item together | `for i, x in enumerate(lst):` | Pairs `(0, first)`, `(1, second)`, ... — replaces a hand-rolled counter |
| `.items()` | Loop over a dict's keys **and** values | `for k, v in d.items():` | Pairs of `(key, value)` |

### Loop control keywords

| Keyword | Effect |
|---|---|
| `break` | Stops the loop immediately — nothing after it runs, even for later items |
| `continue` | Skips the rest of *this* item only, then moves to the next one |

### The core accumulation pattern

The pattern behind almost everything later in the course — including the
whole file organiser — is:

```
result = []
for item in collection:
    if condition(item):
        result.append(item)   # or something derived from item
```

**Loop, decide, collect.** Swap "countries" for "files," and this is
exactly the shape of the folder-organising script.

## 3 — `while` loops

A `while` loop keeps running as long as its condition stays `True`. Unlike
`for`, it doesn't know in advance how many times it will run — so
**something inside the loop must eventually make the condition false**, or
it runs forever (the notebook's stop button interrupts it if that happens).

```
while condition:
    ...              # must change something that condition depends on
```

| Use `for` when... | Use `while` when... |
|---|---|
| You already know what you're looping over (a list, a range, a dict) | You don't know in advance how many iterations you need |
| "Do this once per item" | "Keep doing this until something becomes true" — e.g. rolling a die until you get a 6 |

## 4 — Functions

A function is a named, reusable block of code, defined once with `def` and
called by name as many times as needed.

```
def function_name(parameter_1, parameter_2=default_value):
    ...
    return result   # optional
```

| Concept | Explanation |
|---|---|
| Parameter | A named input the function expects when called |
| Default value (`param=value`) | Used automatically if the caller doesn't supply that argument |
| `return` | Hands a value back to the caller so it can be stored or reused elsewhere |
| `print()` inside a function | Only displays something — it does **not** hand anything back |

### `return` vs `print` — the distinction that matters most

| | `print(x)` | `return x` |
|---|---|---|
| What happens | Shows `x` on screen | Sends `x` back to wherever the function was called |
| Can you reuse the result? | No — nothing is captured | Yes — `result = my_function(...)` stores it |
| Typical use | Debugging, final human-readable output | Passing a value on to more code |

Functions **compose**: one function can call another, each doing one clear
job. That's exactly the shape of `classify()` calling nothing but returning
a category, which a bigger loop then acts on — small pieces, combined.

## 5 — Exploring a folder with `pathlib`

`pathlib` represents a file or folder location as a `Path` **object**,
rather than a plain string — this makes file-handling safe and readable
across operating systems.

### Building and inspecting paths

| Expression | Meaning |
|---|---|
| `Path("sample_data")` | A `Path` pointing at that folder (relative to the notebook) |
| `path_a / "sub" / "file.txt"` | Joins pieces together — the safe alternative to manual slashes |
| `p.name` | Final component: `"WT-DS316-AB-R.txt"` |
| `p.stem` | Name **without** the extension: `"WT-DS316-AB-R"` |
| `p.suffix` | Just the extension: `".txt"` |
| `p.parent` | The containing folder |
| `p.exists()` | `True`/`False` — does anything exist at this path? |
| `p.is_dir()` / `p.is_file()` | Is it a folder / a file? |
| `p.absolute()` | The full path from the drive/filesystem root |

### Listing a folder's contents

| Tool | Returns | Notes |
|---|---|---|
| `data_dir.iterdir()` | A **generator** — one `Path` per entry | Must be wrapped in `list(...)` before slicing (`[0:5]`), since generators can't be sliced directly |
| `list(data_dir.iterdir())` | A real list of `Path` objects | Now indexable, sliceable, loopable like any list |
| `data_dir.glob("*.csv")` | A generator of only the matching paths | `*` means "anything" — `*.csv` matches every CSV directly, no manual `if` needed |

### Why this connects back to Part 1

`.name` (and `.stem`) are just plain strings — every string method and
container pattern from Part A still applies:

- `"asbestos" in f.name.lower()` — substring check
- `f.name.startswith("WT-")` — prefix check
- `f.suffix == ".csv"` — equality check
- looping + counting into a dict, exactly like the `counts_by_suffix` pattern:

```
counts = {}
for f in data_dir.iterdir():
    key = f.suffix
    if key in counts:
        counts[key] = counts[key] + 1
    else:
        counts[key] = 1
```

Nothing new conceptually — just Part A's tools, pointed at real files.

## 6 — Making folders, copying, and writing files

These operations change things on disk, which is why the course convention
is: only ever write into **new** folders you create yourself, never into
`sample_data/`.

| Operation | Syntax | Notes |
|---|---|---|
| Create a folder | `path.mkdir(parents=True, exist_ok=True)` | `parents=True` creates missing parent folders too; `exist_ok=True` means "don't error if it's already there" |
| Copy a file | `shutil.copy(source, destination)` | The destination **folder** must already exist |
| Rename in place | `old_path.rename(new_path)` | Moves/renames; `new_path` is a full `Path`, e.g. built with `.with_name(...)` |
| New path, same folder | `path.with_name("new_name.txt")` | Keeps the parent folder, swaps only the filename |
| Write a file | `path.write_text(some_string)` | Creates the file (or **overwrites** it) with `some_string` |
| Read a file | `path.read_text()` | Returns the whole file as one string |

### The organiser pattern (everything combined)

```
def classify(filename):
    name = filename.lower()
    if <test 1>:
        return "category_1"
    elif <test 2>:
        return "category_2"
    ...
    else:
        return "unclassified"

for f in data_dir.iterdir():
    category = classify(f.name)
    dest_folder = out_root / category
    dest_folder.mkdir(parents=True, exist_ok=True)
    shutil.copy(f, dest_folder / f.name)
```

This is Part B distilled into one script: a **decision function**
(`if`/`elif`/`else` wrapped in `def`, one clear job) combined with a
**loop** (walk every file), acting on each item (`mkdir` + `shutil.copy`),
built entirely from ingredients introduced earlier — nothing here is new.

---

## Quick-reference summary

### Everyday types

| Type | Example | Mutable? |
|---|---|---|
| `int` | `316` | No |
| `float` | `316.0` | No |
| `str` | `"Hello"` | No |
| `bool` | `True` | No |

### Containers

| Container | Ordered? | Changeable? | Duplicates? | Looked up by |
|---|---|---|---|---|
| `list` | Yes | Yes | Yes | Position |
| `set` | No | Yes | No | Membership (`in`) |
| `dict` | Yes | Yes | Keys unique | Key name |

### Operators

| Category | Operators |
|---|---|
| Arithmetic | `+  -  *  /  //  %` |
| Comparison | `==  !=  >  <  >=  <=` |
| Logical | `and  or  not` |

### Control flow & functions

| Keyword / pattern | Purpose |
|---|---|
| `if` / `elif` / `else` | Branch on a condition; first match wins |
| `for item in collection:` | Loop over a known collection |
| `while condition:` | Loop until a condition becomes false |
| `break` / `continue` | Exit the loop entirely / skip to the next item |
| `def name(params):` / `return` | Define a reusable, callable block that can hand back a result |

### `pathlib` / file operations

| Task | Tool |
|---|---|
| Point at a location | `Path("folder") / "file.txt"` |
| Inspect it | `.name` `.stem` `.suffix` `.parent` `.exists()` `.is_dir()` `.is_file()` |
| List a folder | `list(path.iterdir())` or `path.glob("pattern")` |
| Make a folder | `path.mkdir(parents=True, exist_ok=True)` |
| Copy / rename | `shutil.copy(src, dst)` / `old.rename(new)` |
| Write / read text | `path.write_text(s)` / `path.read_text()` |